# Comparación entre Gini y Entropía
## Variable categórica: `estado_civil_desc`

Se calculan sobre la misma variable:

1. **Entropía de Shannon** (heterogeneidad).
2. **Coeficiente de Gini** aplicado a las frecuencias de las categorías (concentración).

Todo se calcula ponderado con `factor_expansion` (población) y sin ponderar (muestra), para todas las mujeres y solo para las que reportaron violencia de pareja.

## 1. Importaciones y configuración

In [5]:
import sys
from pathlib import Path

import numpy as np
import polars as pl
import matplotlib.pyplot as plt

# Estilo de gráficas
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
})

# ---- Nombres de columnas que vamos a usar ----
VAR   = "estado_civil_desc"          # variable categórica elegida
PESO  = "factor_expansion"           # factor de expansión (FAC_MUJ)
VIOL  = "sufrio_violencia_pareja"    # 1 = sí sufrió violencia de pareja, 0 = no

print("Polars:", pl.__version__, "| NumPy:", np.__version__)

Polars: 1.44.2 | NumPy: 2.5.3


## 2. Carga de datos

In [ ]:
# la raíz del proyecto se agrega al path y la ruta viene de config/rutas.py
sys.path.append(str(Path.cwd().parent))

try:
    from config.rutas import ARCHIVO_PROCESSED
    RUTA_DATOS = ARCHIVO_PROCESSED
except ImportError:                      
    RUTA_DATOS = Path("endireh_2021_clean.csv")

FIG_DIR = Path("figuras")      # carpeta de salida de las figuras
RES_DIR = Path(".")            # carpeta de salida del CSV de resultados
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("Ruta de datos:", RUTA_DATOS)

FileNotFoundError: No se encontró /Users/julifm/septimo/proyecto-endireh-violencia/data/data-processed/endireh_2021_clean.csv. Revisa ARCHIVO_PROCESSED en config/rutas.py.

In [ ]:
df = pl.read_csv(RUTA_DATOS)

print(f"Filas: {df.height:,} | Columnas: {df.width}")
df.head()

In [ ]:
faltantes = [c for c in (VAR, PESO, VIOL) if c not in df.columns]
assert not faltantes, f"Faltan columnas en el dataset: {faltantes}. Columnas disponibles: {df.columns}"
print("Las tres columnas necesarias están presentes")

## 3. Revisión de variables

In [ ]:
print("=== estado_civil_desc ===")
print("Categorías distintas:", df[VAR].n_unique())
print("Nulos:", df[VAR].null_count())
print(df[VAR].unique().sort().to_list())

In [ ]:
print("=== sufrio_violencia_pareja ===")
print("Valores distintos:", df[VIOL].unique().sort().to_list())
print("Nulos:", df[VIOL].null_count())
print()
print("=== factor_expansion ===")
print("Nulos:", df[PESO].null_count())
print("Valores <= 0:", df.filter(pl.col(PESO) <= 0).height)
print("Mínimo / máximo:", df[PESO].min(), "/", df[PESO].max())
print("Población total representada (suma de factores): {:,.0f}".format(df[PESO].sum()))

## 4. Funciones

- Entropía: $H = -\sum p_i \log_2 p_i$, normalizada $H_{norm} = H/\log_2 k$.
- Gini sobre frecuencias: $G = \dfrac{2\sum i\,x_{(i)} - (k+1)\sum x_{(i)}}{k\sum x_{(i)}}$, normalizado $G_{norm} = G\cdot\dfrac{k}{k-1}$.
- Gini-Simpson: $GS = 1-\sum p_i^2$. IQV: $k\,GS/(k-1)$.

In [ ]:
def shannon(p):
    '''Entropía de Shannon en bits. Recibe un arreglo de proporciones que suman 1.'''
    p = np.asarray(p, dtype=float)
    p = p[p > 0]                      # convención 0 log(0) = 0
    return float(-(p * np.log2(p)).sum()) + 0.0   # + 0.0 evita mostrar -0.0


def gini(x):
    '''Coeficiente de Gini (fórmula de la práctica). Recibe frecuencias o cantidades >= 0.'''
    x = np.sort(np.asarray(x, dtype=float))   # orden ascendente
    n = len(x)
    i = np.arange(1, n + 1)
    return float((2 * (i * x).sum() - (n + 1) * x.sum()) / (n * x.sum()))


def gini_simpson(p):
    '''Índice de Gini-Simpson: 1 - suma(p^2).'''
    p = np.asarray(p, dtype=float)
    return float(1 - (p ** 2).sum())


def iqv(p):
    '''Índice de Variación Cualitativa: Gini-Simpson normalizado a [0, 1].'''
    k = len(p)
    return float(k * gini_simpson(p) / (k - 1))


def indices_desde_frecuencias(n):
    '''
    A partir de las frecuencias (ponderadas o no) de cada categoría,
    calcula todos los índices y los devuelve en un diccionario.
    '''
    n = np.asarray(n, dtype=float)
    p = n / n.sum()
    k = len(n)
    H = shannon(p)
    G = gini(n)
    return {
        "k": k,
        "H (bits)": H,
        "H max (bits)": float(np.log2(k)),
        "H_norm": H / np.log2(k),
        "G": G,
        "G max": (k - 1) / k,
        "G_norm": G * k / (k - 1),
        "Gini-Simpson": gini_simpson(p),
        "IQV": iqv(p),
    }


def frecuencias(datos, var, peso=None):
    '''
    Tabla de frecuencias de una variable categórica con polars.
      - peso=None  -> cuenta filas (describe a la MUESTRA)
      - peso="col" -> suma el factor de expansión (describe a la POBLACIÓN)
    Devuelve columnas: var, n, p (proporción), ordenadas de mayor a menor.
    '''
    if peso is None:
        agregacion = pl.len().cast(pl.Float64).alias("n")
    else:
        agregacion = pl.col(peso).cast(pl.Float64).sum().alias("n")
    f = datos.group_by(var).agg(agregacion)
    return (f.with_columns((pl.col("n") / pl.col("n").sum()).alias("p"))
             .sort("p", descending=True))

print("Funciones definidas ")

## 5. Prueba de cordura (casos extremos)

In [ ]:
casos = {
    "Parejo (25/25/25/25)":      [25, 25, 25, 25],
    "Domina una (97/1/1/1)":     [97, 1, 1, 1],
    "Todo en una (100/0/0/0)":   [100, 0, 0, 0],
}

for nombre, n in casos.items():
    r = indices_desde_frecuencias(n)
    print(f"{nombre:28s} H={r['H (bits)']:.3f}  H_norm={r['H_norm']:.3f}  G={r['G']:.3f}  G_norm={r['G_norm']:.3f}")

# Comprobaciones automáticas 
r = indices_desde_frecuencias([25, 25, 25, 25])
assert abs(r["H (bits)"] - 2.0) < 1e-9 and abs(r["H_norm"] - 1.0) < 1e-9
assert abs(r["G"]) < 1e-9 and abs(r["G_norm"]) < 1e-9

r = indices_desde_frecuencias([100, 0, 0, 0])
assert abs(r["H (bits)"]) < 1e-9
assert abs(r["G"] - 0.75) < 1e-9 and abs(r["G_norm"] - 1.0) < 1e-9

print("\nTodas las pruebas de cordura pasaron ")

## 6. Distribución de `estado_civil_desc`

In [ ]:
df_viol = df.filter(pl.col(VIOL) == 1)
print(f"Todas las mujeres (filas):                {df.height:,}")
print(f"Mujeres con violencia de pareja (filas):  {df_viol.height:,}  ({df_viol.height/df.height:.1%} de la muestra)")

In [ ]:
f_todas_pond = frecuencias(df,      VAR, PESO)
f_todas_sin  = frecuencias(df,      VAR, None)
f_viol_pond  = frecuencias(df_viol, VAR, PESO)
f_viol_sin   = frecuencias(df_viol, VAR, None)

print("=== TODAS las mujeres · PONDERADO (población) ===")
f_todas_pond

Ponderada vs. sin ponderar (todas las mujeres):

In [ ]:
# Tabla comparativa de proporciones (ponderada vs. sin ponderar) para todas las mujeres
comparacion = (
    f_todas_pond.select([VAR, pl.col("p").alias("p_ponderada")])
    .join(f_todas_sin.select([VAR, pl.col("p").alias("p_sin_ponderar")]), on=VAR)
    .with_columns((pl.col("p_ponderada") - pl.col("p_sin_ponderar")).alias("diferencia"))
    .sort("p_ponderada", descending=True)
)
comparacion

Ponderada vs. sin ponderar (con violencia de pareja):

In [ ]:
# Misma comparación para las mujeres que reportaron violencia de pareja
comparacion_viol = (
    f_viol_pond.select([VAR, pl.col("p").alias("p_ponderada")])
    .join(f_viol_sin.select([VAR, pl.col("p").alias("p_sin_ponderar")]), on=VAR, how="left")
    .fill_null(0.0)
    .sort("p_ponderada", descending=True)
)
comparacion_viol

## 7. Cálculo de índices

In [ ]:
escenarios = {
    "Todas · ponderado":          f_todas_pond,
    "Todas · sin ponderar":       f_todas_sin,
    "Con violencia · ponderado":  f_viol_pond,
    "Con violencia · sin ponderar": f_viol_sin,
}

resultados = {}
for nombre, f in escenarios.items():
    resultados[nombre] = indices_desde_frecuencias(f["n"].to_numpy())

claves = ["k", "H (bits)", "H max (bits)", "H_norm", "G", "G max", "G_norm", "Gini-Simpson", "IQV"]
tabla = pl.DataFrame({
    "Índice": claves,
    **{nombre: [round(float(r[c]), 4) for c in claves] for nombre, r in resultados.items()},
})
tabla

In [ ]:
# Ver explícitamente la relación entre ambos índices en cada escenario
print(f"{'Escenario':32s} {'H_norm':>8s} {'G_norm':>8s} {'H_norm+G_norm':>15s}")
for nombre, r in resultados.items():
    print(f"{nombre:32s} {r['H_norm']:8.3f} {r['G_norm']:8.3f} {r['H_norm'] + r['G_norm']:15.3f}")

## 8. Visualizaciones

**Figura A.** Distribución del estado civil.

In [ ]:
# ---------- Figura A: proporción por estado civil ----------
etq = f_todas_pond[VAR].to_list()                       # orden de mayor a menor (todas, ponderado)
p_todas = dict(zip(f_todas_pond[VAR].to_list(), f_todas_pond["p"].to_list()))
p_viol  = dict(zip(f_viol_pond[VAR].to_list(),  f_viol_pond["p"].to_list()))

x = np.arange(len(etq))
ancho = 0.38

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(x - ancho/2, [p_todas.get(e, 0) * 100 for e in etq], ancho, label="Todas las mujeres", color="#4C72B0")
ax.bar(x + ancho/2, [p_viol.get(e, 0)  * 100 for e in etq], ancho, label="Con violencia de pareja reportada", color="#DD8452")
ax.set_xticks(x)
ax.set_xticklabels(etq, rotation=20, ha="right")
ax.set_ylabel("% de la población representada")
ax.set_xlabel("Estado civil")
ax.set_title("Figura A. Distribución del estado civil (ponderada por factor de expansión)")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "figA_distribucion_estado_civil.png", bbox_inches="tight")
plt.show()

**Figura B.** Entropía vs. Gini.

In [ ]:
# ---------- Figura B: H_norm vs G_norm por escenario ----------
nombres = list(resultados.keys())
H_n = [resultados[n]["H_norm"] for n in nombres]
G_n = [resultados[n]["G_norm"] for n in nombres]

x = np.arange(len(nombres))
ancho = 0.38

fig, ax = plt.subplots(figsize=(9, 4.5))
b1 = ax.bar(x - ancho/2, H_n, ancho, label="Entropía normalizada (heterogeneidad)", color="#55A868")
b2 = ax.bar(x + ancho/2, G_n, ancho, label="Gini normalizado (concentración)", color="#C44E52")
for barras in (b1, b2):
    for b in barras:
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.015, f"{b.get_height():.2f}", ha="center", fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(nombres, rotation=15, ha="right")
ax.set_ylim(0, 1.12)
ax.set_ylabel("Valor del índice (escala 0 a 1)")
ax.set_title("Figura B. Entropía vs. Gini sobre la misma variable (estado civil)")
ax.legend(loc="upper center", ncol=1, fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / "figB_entropia_vs_gini.png", bbox_inches="tight")
plt.show()

**Figura C.** Curva de Lorenz de las frecuencias.

In [ ]:
# ---------- Figura C: curva de Lorenz de las frecuencias de las categorías ----------
def curva_lorenz(n):
    '''Devuelve (proporción acumulada de categorías, proporción acumulada de casos).'''
    x = np.sort(np.asarray(n, dtype=float))
    acum = np.cumsum(x) / x.sum()
    return np.insert(np.linspace(1/len(x), 1, len(x)), 0, 0), np.insert(acum, 0, 0)

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.plot([0, 1], [0, 1], "--", color="gray", label="Igualdad perfecta (G = 0)")
for nombre, color in [("Todas · ponderado", "#4C72B0"), ("Con violencia · ponderado", "#DD8452")]:
    xs, ys = curva_lorenz(escenarios[nombre]["n"].to_numpy())
    ax.plot(xs, ys, marker="o", color=color, label=f"{nombre} (G = {resultados[nombre]['G']:.3f})")
ax.set_xlabel("Proporción acumulada de categorías (de menor a mayor frecuencia)")
ax.set_ylabel("Proporción acumulada de casos")
ax.set_title("Figura C. Curva de Lorenz de las frecuencias\nde estado civil")
ax.set_aspect("equal")
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.savefig(FIG_DIR / "figC_lorenz_estado_civil.png", bbox_inches="tight")
plt.show()

## 9. Interpretación

In [ ]:
for nombre, f in escenarios.items():
    r = resultados[nombre]
    top = f.row(0, named=True)
    print(f"■ {nombre}")
    print(f"   Categoría más grande: {top[VAR]} ({top['p']:.1%})")
    print(f"   H_norm = {r['H_norm']:.3f}  |  G_norm = {r['G_norm']:.3f}")
    print()

## 10. Guardado de resultados

In [ ]:
archivo = RES_DIR / "resultados_gini_entropia_estado_civil.csv"
tabla.write_csv(archivo)
print("Resultados guardados en:", archivo)